# Transformer-Based Classification for BI-RADS

Fine-tune BERTimbau (Portuguese BERT) for BI-RADS classification using contextual embeddings.  
Compare against the TF-IDF baseline to quantify the benefit of semantic text understanding.

In [ ]:
RANDOM_STATE = 42
N_FOLDS = 5
METRIC = "f1_macro"
DATA_PATH = "data/raw/train.csv"
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
LOSS_TYPE = "focal"
FOCAL_GAMMA = 2.0
FREEZE_LAYERS = 0
MLFLOW_EXPERIMENT = "birads-transformer"

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
    AdamW,
)

matplotlib.use("Agg")
warnings.filterwarnings("ignore")

COMPETITION = "spr-2026-mammography-report-classification"

KAGGLE_INPUT = Path("/kaggle/input") / COMPETITION
if KAGGLE_INPUT.exists():
    DATA_PATH = str(KAGGLE_INPUT / "train.csv")
    IS_KAGGLE = True
else:
    IS_KAGGLE = False


def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    return Path.cwd()


ROOT = _find_project_root()
_data_path = Path(DATA_PATH)
if not _data_path.is_absolute():
    _data_path = ROOT / _data_path
DATA_PATH = str(_data_path)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print(f"Project root: {ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Kaggle env: {IS_KAGGLE}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## MLflow Initialization

In [ ]:
import mlflow
from src.config.mlflow_init import init_mlflow

mlflow_client = init_mlflow(MLFLOW_EXPERIMENT)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

## Data Loading and Tokenization

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} reports.")
df.head()

In [ ]:
class BIRADSdataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            return_token_type_ids=False,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "text": text,
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

## Focal Loss Implementation

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction="none", weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss

## Training Loop

In [ ]:
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0

    for d in tqdm(data_loader, desc="Training"):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        _, preds = torch.max(logits, dim=1)
        loss = loss_fn(logits, labels)

        correct_predictions += torch.sum(preds == labels)
        losses.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / n_examples, np.mean(losses)


def eval_model(model, data_loader, loss_fn, device, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for d in tqdm(data_loader, desc="Evaluating"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            _, preds = torch.max(logits, dim=1)
            loss = loss_fn(logits, labels)

            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return (
        correct_predictions.double() / n_examples,
        np.mean(losses),
        f1,
        all_preds,
        all_labels,
    )

## Main Execution: 5-Fold Cross-Validation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
X = df["report"].values
y = df["target"].values

fold_f1_scores = []
oof_preds = np.zeros(len(df))
oof_probs = np.zeros((len(df), 7))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")

    df_train = df.iloc[train_idx]
    df_val = df.iloc[val_idx]

    train_dataset = BIRADSdataset(
        texts=df_train.report.to_numpy(),
        labels=df_train.target.to_numpy(),
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )
    val_dataset = BIRADSdataset(
        texts=df_val.report.to_numpy(),
        labels=df_val.target.to_numpy(),
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=7
    ).to(device)

    if FREEZE_LAYERS > 0:
        # Simplified freezing logic for BERT-like models
        if hasattr(model, "bert"):
            for layer in model.bert.encoder.layer[:FREEZE_LAYERS]:
                for param in layer.parameters():
                    param.requires_grad = False
        elif hasattr(model, "roberta"):
            for layer in model.roberta.encoder.layer[:FREEZE_LAYERS]:
                for param in layer.parameters():
                    param.requires_grad = False

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )

    if LOSS_TYPE == "focal":
        loss_fn = FocalLoss(gamma=FOCAL_GAMMA).to(device)
    else:
        # Class weighted CE
        class_weights = torch.tensor(
            [1.0 / (df_train.target == i).mean() for i in range(7)], dtype=torch.float
        ).to(device)
        class_weights = class_weights / class_weights.sum() * 7
        loss_fn = nn.CrossEntropyLoss(weight=class_weights).to(device)

    best_f1 = 0
    patience = 2
    trigger_times = 0

    for epoch in range(NUM_EPOCHS):
        train_acc, train_loss = train_epoch(
            model, train_loader, loss_fn, optimizer, device, scheduler, len(df_train)
        )
        val_acc, val_loss, val_f1, preds, labels = eval_model(
            model, val_loader, loss_fn, device, len(df_val)
        )

        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        print(f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            trigger_times = 0
            # Save best for this fold if needed
            # torch.save(model.state_index(), f"best_model_fold_{fold}.bin")
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("Early stopping!")
                break

    fold_f1_scores.append(best_f1)

    # Final evaluation for OOF
    _, _, _, preds, labels = eval_model(model, val_loader, loss_fn, device, len(df_val))
    oof_preds[val_idx] = preds

    # Get probabilities for OOF
    model.eval()
    probs = []
    with torch.no_grad():
        for d in val_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs.extend(F.softmax(outputs.logits, dim=1).cpu().numpy())
    oof_probs[val_idx] = probs

print(
    f"\nMean 5-fold CV macro-F1: {np.mean(fold_f1_scores):.4f} (+/- {np.std(fold_f1_scores):.4f})"
)

## Final Evaluation and MLflow Logging

In [ ]:
with mlflow.start_run():
    mlflow.log_params(
        {
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "num_epochs": NUM_EPOCHS,
            "warmup_ratio": WARMUP_RATIO,
            "weight_decay": WEIGHT_DECAY,
            "loss_type": LOSS_TYPE,
            "focal_gamma": FOCAL_GAMMA,
            "freeze_layers": FREEZE_LAYERS,
            "random_state": RANDOM_STATE,
            "n_folds": N_FOLDS,
            "feature_set_version": "v3.0",
        }
    )

    mlflow.log_metric("cv_f1_macro", np.mean(fold_f1_scores))

    # Log per-class metrics
    report = classification_report(y, oof_preds, output_dict=True, zero_division=0)
    for i in range(7):
        if str(i) in report:
            mlflow.log_metric(f"f1_class_{i}", report[str(i)]["f1-score"])

    print("\nOverall OOF Classification Report:")
    print(classification_report(y, oof_preds, zero_division=0))

    # Confusion Matrix Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay.from_predictions(y, oof_preds, ax=ax, cmap="Blues")
    plt.title(f"Confusion Matrix - {MODEL_NAME}")
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.show()

## Kaggle Runtime Profiling

In [ ]:
# This cell estimates the total runtime for Kaggle
# 5 folds of training + inference
import time

print("--- Runtime Estimate ---")
if "fold_f1_scores" in locals() and len(fold_f1_scores) > 0:
    print(
        "Actual training time should be tracked within the loop for precise estimation."
    )
else:
    print("Run the CV loop first to get timing data.")